In [ ]:
import gdown
import zipfile
file_id = '1eysSxAcT4zBNLl2D_eK-OPwhGSjemskS'
output_path = 'downloaded_file.zip'
gdown.download(f'https://drive.google.com/uc?id={file_id}', output_path, fuzzy=True)

# Extract the zip file
with zipfile.ZipFile(output_path, 'r') as zip_ref:
    zip_ref.extractall('.')

print("File downloaded and extracted successfully.")

Downloading...
From: https://drive.google.com/uc?id=1eysSxAcT4zBNLl2D_eK-OPwhGSjemskS
To: /content/downloaded_file.zip
100%|██████████| 21.3M/21.3M [00:00<00:00, 47.9MB/s]


File downloaded and extracted successfully.


In [ ]:
#import of MD data
import scipy.io
import numpy as np
matMD59 = scipy.io.loadmat('/content/MD_m59Data.mat')['MD_m59Data']
matMDSoil=scipy.io.loadmat('/content/MD_SoilData.mat')['MD_SoilData']
matMDnonmine=scipy.io.loadmat('/content/matAll.mat')['matMDnonmine']
matMDSoilClutter=np.concatenate((matMDSoil, matMDnonmine), axis=2)
matMD=np.concatenate((matMD59, matMDSoilClutter), axis=2)

# Rotate data to make it 264*11*10 instead of 11*10*264
# First 108 entries correspond to presence of mine
# Remaining entries (up to 264) correspond to clutter and/or soil
matMD = np.rot90(matMD,1,(0,2))
matMD = np.rot90(matMD,1,(1,2))
matMD = np.fliplr(matMD)

# Labeling the data
# The first 108 belong to the Mine class (0).
# The rest belong to the harmless class (soil and clutter).
noSamples=matMD.shape[0]
mat_labels = np.zeros(noSamples)
mat_labels[108:] = 1  # Set elements from index 108 to the end to 1

# Augment the data using flipping
matMD1 = np.flip(matMD,1)
matMD2 = np.flip(matMD,2)
matMD12 = np.flip(np.flip(matMD,1),2)
matMD = np.concatenate((matMD,matMD1,matMD2,matMD12), axis=0)
# Resize the labels vector after data augmentation
mat_labels = np.concatenate((mat_labels,mat_labels,mat_labels,mat_labels), axis=0)
# Flatten data to make it 1056*110
matMD = matMD.reshape(-1, 11*10).astype('float32')

# Function to add the features to the data matrix
def addFeatures(dataMat):
	# mean, min, max and std of the original data
	dataMean = np.mean(dataMat, axis=1).reshape(-1,1)
	dataMin = np.min(dataMat, axis=1).reshape(-1,1)
	dataMax = np.max(dataMat, axis=1).reshape(-1,1)
	dataStd = np.std(dataMat, axis=1).reshape(-1,1)
	data_with_Features = np.concatenate((dataMat, dataMean, dataMin, dataMax, dataStd), axis=1)
	# Difference matrix and its mean, min, max and std
	data_Dif = np.diff(dataMat,axis=1)
	data_Dif_Mean = np.mean(data_Dif, axis=1).reshape(-1,1)
	data_Dif_Min = np.min(data_Dif, axis=1).reshape(-1,1)
	data_Dif_Max = np.max(data_Dif, axis=1).reshape(-1,1)
	data_Dif_Std = np.std(data_Dif, axis=1).reshape(-1,1)
	data_with_Features = np.concatenate((data_with_Features, data_Dif, data_Dif_Mean, data_Dif_Min, data_Dif_Max, data_Dif_Std), axis=1)
	# return data with added features
	return data_with_Features

  # Add features and check new dimensions
matMD_with_Features = addFeatures(matMD)
# zzzzzzzzzzzz
# Generate Gaussian noise
mean = 193
std_dev = 50
gaussian_noise = np.random.normal(mean, std_dev, size=matMD.shape)
# Generate uniform noise
low = 0  # Lower bound of the noise
high = 385  # Upper bound of the noise
uniform_noise = np.random.uniform(low, high, size=matMD.shape)

# Create the noisy array
gaussian_array = matMD #+ gaussian_noise
uniform_array = matMD #+ uniform_noise

gaussian_array_with_Features = addFeatures(gaussian_array)
uniform_array_with_Features = addFeatures(uniform_array)


data_set = matMD_with_Features
labels = mat_labels
# Add the noisy versions
MD_data_set = np.concatenate((matMD_with_Features,gaussian_array_with_Features,uniform_array_with_Features),axis=0)
MD_labels = np.concatenate((mat_labels, mat_labels, mat_labels), axis=0)

#import of GPR data
import scipy.io
import numpy as np


gprSoil = scipy.io.loadmat('/content/GPR_SoilData.mat')['GPR_SoilData']
gpr59=scipy.io.loadmat('/content/GPR_m59Data.mat')['GPR_m59Data']
gprClutter=scipy.io.loadmat('/content/GPR_ClutterData.mat')['GPR_ClutterData']
matGPR=np.concatenate((gpr59, gprSoil, gprClutter), axis=3)
matGPR = np.rot90(matGPR,1,(0,3))
matGPR = np.rot90(matGPR,1,(1,3))
matGPR = np.rot90(matGPR,1,(2,3))
matGPR = np.fliplr(matGPR)
# 0 is for class Landmine (danger present)
# 1 is for class Soil
# 2 is for class Clutter
labels = np.zeros(264)
labels[108:216]=1
labels[216:]=2

def getPulseDerivative(data):
     dataD=data.copy()
     for t in range(1,data.shape[3]):
          dataD[:,:,:,t]=data[:,:,:,t]-data[:,:,:,t-1]
     dataD=dataD[:,:,:,1:]
     return dataD

def getPulseFeatures(data,func):
     features = []
     for n in range(data.shape[0]):
          for m in range(data.shape[1]):
               features.append(func(data[n,m,:]))
     features = np.array(features)
     return features

def getHistogramData(data,bins=4):
     hist = np.histogram(data,bins)
     hist = hist[0]
     return hist

def getAllFeatures(data):
    FeaturesData = np.empty(shape=[0,632])              # features of a version of the dataset
    dataD = getPulseDerivative(data)
    for r in range(data.shape[0]):
        features = []                               # features of each record (measurement) of size 11x7x512
        hist = getHistogramData(data[r,:,:,:])
        histD = getHistogramData(dataD[r,:,:,:])
        maxAll = np.max(data[r,:,:,:])
        minAll = np.min(data[r,:,:,:])
        meanAll = np.mean(data[r,:,:,:])
        stdAll = np.std(data[r,:,:,:])
        maxAllD = np.max(dataD[r,:,:,:])
        minAllD = np.min(dataD[r,:,:,:])
        meanAllD = np.mean(dataD[r,:,:,:])
        stdAllD = np.std(dataD[r,:,:,:])
        features = [maxAll,maxAllD,minAll,minAllD,meanAll,meanAllD,stdAll,stdAllD]
        features = np.concatenate((features, hist, histD))
        features = np.concatenate((features,getPulseFeatures(data[r,:,:,:], np.max)))
        features = np.concatenate((features,getPulseFeatures(data[r,:,:,:], np.min)))
        features = np.concatenate((features,getPulseFeatures(data[r,:,:,:], np.mean)))
        features = np.concatenate((features,getPulseFeatures(data[r,:,:,:], np.std)))
        features = np.concatenate((features,getPulseFeatures(dataD[r,:,:,:], np.max)))
        features = np.concatenate((features,getPulseFeatures(dataD[r,:,:,:], np.min)))
        features = np.concatenate((features,getPulseFeatures(dataD[r,:,:,:], np.mean)))
        features = np.concatenate((features,getPulseFeatures(dataD[r,:,:,:], np.std)))
        features = features.reshape(1,-1)
        FeaturesData = np.concatenate((FeaturesData,features),axis=0)

    return FeaturesData

# Augment the data with a noisy version
matGPR_UniformNoise = matGPR.copy()
matGPR_GaussianNoise = matGPR.copy()
for n in range(matGPR.shape[0]):
	uniform_noise = np.random.uniform(0.95, 1.05, size=matGPR.shape[1:3])
	matGPR_UniformNoise[n,:,:,:] = matGPR_UniformNoise[n,:,:,:]*uniform_noise[:,:,np.newaxis]
	gaussian_noise = np.random.normal(1, 0.1, size=matGPR.shape[1:3])
	matGPR_GaussianNoise[n,:,:,:] = matGPR_GaussianNoise[n,:,:,:]*gaussian_noise[:,:,np.newaxis]

matGPR = np.concatenate((matGPR,matGPR_UniformNoise, matGPR_GaussianNoise), axis=0)
labels = np.concatenate((labels, labels, labels), axis=0)

# Augment the data using flipping
matGPR1 = np.flip(matGPR,1)
matGPR2 = np.flip(matGPR,2)
matGPR12 = np.flip(np.flip(matGPR,1),2)
matGPR = np.concatenate((matGPR,matGPR1,matGPR2,matGPR12), axis=0)
GPR_with_Features = getAllFeatures(matGPR)

GPR_data_set = GPR_with_Features
GPR_labels = np.concatenate((labels, labels, labels, labels), axis=0)

print('Shape of the MD_features data is', MD_data_set.shape)
print('Shape of the GPR_ features data is',GPR_data_set.shape)

print('Shape of MD labels', MD_labels.shape)
print('Shape of GPR labels', GPR_labels.shape)
# zzzzzzzzzzz
combined_array = np.concatenate((MD_data_set, GPR_data_set), axis=1)
combined_array.shape

labels = np.concatenate((labels, labels, labels, labels), axis=0)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
X = combined_array
y = labels
print("X Shape:", X.shape)
print("y Shape:", y.shape)
# train_test_split(
#     matGPR_scaled, matMDall_scaled, labels_cat, test_size=0.2, random_state=42, stratify=labels
# )
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=labels)
print('Shape of the training data is', X_train.shape)
print('Shape of the testing data is', X_test.shape)


print(matGPR.shape)
print(matMD.shape)
print(gaussian_array.shape)
print(uniform_array.shape)
print(labels.shape)

matMDall=np.concatenate((matMD.reshape(-1, 11,10), gaussian_array.reshape(-1, 11,10), uniform_array.reshape(-1, 11,10)), axis=0)
print(matMDall.shape)

Shape of the MD_features data is (3168, 227)
Shape of the GPR_ features data is (3168, 632)
Shape of MD labels (3168,)
Shape of GPR labels (3168,)
X Shape: (3168, 859)
y Shape: (3168,)
Shape of the training data is (2534, 859)
Shape of the testing data is (634, 859)
(3168, 11, 7, 512)
(1056, 110)
(1056, 110)
(1056, 110)
(3168,)
(3168, 11, 10)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

def normalize_confusion_matrix(cm):
    """Normalizes confusion matrix rows to sum to 1 (percentages)."""
    # Handle cases where cm might be empty or sums to zero to prevent division by zero
    if cm.size == 0 or np.sum(cm.sum(axis=1)) == 0:
        return np.array([[]])
    # Ensure sum(axis=1) has the same number of dimensions as cm to avoid broadcasting issues
    row_sums = cm.sum(axis=1)
    row_sums = np.where(row_sums == 0, 1e-10, row_sums) # Replace 0 with a small number to avoid division by zero
    return cm.astype('float') / row_sums[:, np.newaxis]

X=combined_array.copy()
y=labels.copy()

# Initialize StratifiedKFold
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# --- Logistic Regression ---
print('\n\nLogistic Regression with 3-fold Stratified Cross-Validation')

accuracy_lr_scores = []
f1_lr_scores = []
confusion_matrices_lr = []

for fold, (train_index, test_index) in enumerate(skf.split(X, y)):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Handle class weights if applicable (re-compute for each fold if needed or use a robust approach)
    classes = np.unique(y_train)
    weights = compute_class_weight('balanced', classes=classes, y=y_train)
    class_weights_lr = dict(zip(classes, weights))

    modelLR = LogisticRegression(class_weight=class_weights_lr, max_iter=10000)
    modelLR.fit(X_train, y_train)
    y_pred_LR = modelLR.predict(X_test)

    accuracy_lr_scores.append(accuracy_score(y_test, y_pred_LR))
    f1_lr_scores.append(f1_score(y_test, y_pred_LR, average='macro'))
    confusion_matrices_lr.append(confusion_matrix(y_test, y_pred_LR))

print(f"Average GPR-MD Accuracy with Logistic Regression: {np.mean(accuracy_lr_scores):.4f}")
print(f"Average MD F1 score with Logistic Regression: {np.mean(f1_lr_scores):.4f}")
print("Average MD Confusion Matrix with Logistic Regression (sum over folds):")
summed_cm_lr = np.sum(confusion_matrices_lr, axis=0)
print(summed_cm_lr)
print("Normalized MD Confusion Matrix with Logistic Regression (sum over folds):")
print(normalize_confusion_matrix(summed_cm_lr).round(2))

# --- Random Forest ---
print('\n\nRandom Forest with 5-fold Stratified Cross-Validation')

accuracy_rf_scores = []
f1_rf_scores = []
confusion_matrices_rf = []

for fold, (train_index, test_index) in enumerate(skf.split(X, y)):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Handle class weights
    classes = np.unique(y_train)
    weights = compute_class_weight('balanced', classes=classes, y=y_train)
    class_weights_rf = dict(zip(classes, weights))

    modelRF = RandomForestClassifier(class_weight=class_weights_rf, n_estimators=100, random_state=42)
    modelRF.fit(X_train, y_train)
    y_pred_RF = modelRF.predict(X_test)

    accuracy_rf_scores.append(accuracy_score(y_test, y_pred_RF))
    f1_rf_scores.append(f1_score(y_test, y_pred_RF, average='macro'))
    confusion_matrices_rf.append(confusion_matrix(y_test, y_pred_RF))

print(f"Average GPR-MD Accuracy with Random Forest: {np.mean(accuracy_rf_scores):.4f}")
print(f"Average MD F1 score with Random Forest: {np.mean(f1_rf_scores):.4f}")
print("Average MD Confusion Matrix with Random Forest (sum over folds):")
summed_cm_rf = np.sum(confusion_matrices_rf, axis=0)
print(summed_cm_rf)
print("Normalized MD Confusion Matrix with Random Forest (sum over folds):")
print(normalize_confusion_matrix(summed_cm_rf).round(2))

# --- XGBoost ---
print('\n\nXGBoost with 5-fold Stratified Cross-Validation')

accuracy_xgb_scores = []
f1_xgb_scores = []
confusion_matrices_xgb = []

for fold, (train_index, test_index) in enumerate(skf.split(X, y)):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # XGBoost can handle class imbalance directly or via sample weights
    # For simplicity, will not pass class_weights directly, but it can be done via sample_weight in fit or `scale_pos_weight` parameter.
    modelXGB = XGBClassifier(random_state=42)
    modelXGB.fit(X_train, y_train)
    y_pred_XGB = modelXGB.predict(X_test)

    accuracy_xgb_scores.append(accuracy_score(y_test, y_pred_XGB))
    f1_xgb_scores.append(f1_score(y_test, y_pred_XGB, average='macro'))
    confusion_matrices_xgb.append(confusion_matrix(y_test, y_pred_XGB))

print(f"Average GPR-MD Accuracy with XGBoost: {np.mean(accuracy_xgb_scores):.4f}")
print(f"Average MD F1 score with XGBoost: {np.mean(f1_xgb_scores):.4f}")
print("Average MD Confusion Matrix with XGBoost (sum over folds):")
summed_cm_xgb = np.sum(confusion_matrices_xgb, axis=0)
print(summed_cm_xgb)
print("Normalized MD Confusion Matrix with XGBoost (sum over folds):")
print(normalize_confusion_matrix(summed_cm_xgb).round(2))



Logistic Regression with 5-fold Stratified Cross-Validation


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

Average GPR-MD Accuracy with Logistic Regression: 0.8681
Average MD F1 score with Logistic Regression: 0.8344
Average MD Confusion Matrix with Logistic Regression (sum over folds):
[[1259   22   15]
 [  24 1088  184]
 [  15  158  403]]
Normalized MD Confusion Matrix with Logistic Regression (sum over folds):
[[0.97 0.02 0.01]
 [0.02 0.84 0.14]
 [0.03 0.27 0.7 ]]


Random Forest with 5-fold Stratified Cross-Validation
Average GPR-MD Accuracy with Random Forest: 0.8759
Average MD F1 score with Random Forest: 0.8156
Average MD Confusion Matrix with Random Forest (sum over folds):
[[1296    0    0]
 [   1 1199   96]
 [   0  296  280]]
Normalized MD Confusion Matrix with Random Forest (sum over folds):
[[1.   0.   0.  ]
 [0.   0.93 0.07]
 [0.   0.51 0.49]]


XGBoost with 5-fold Stratified Cross-Validation
Average GPR-MD Accuracy with XGBoost: 0.9741
Average MD F1 score with XGBoost: 0.9657
Average MD Confusion Matrix with XGBoost (sum over folds):
[[1294    2    0]
 [   0 1264   32]
 [   0 

In [ ]:
import pandas as pd
import numpy as np

def normalize_confusion_matrix(cm):
    """Normalizes confusion matrix rows to sum to 1 (percentages)."""
    # Handle cases where cm might be empty or sums to zero to prevent division by zero
    if cm.size == 0 or np.sum(cm.sum(axis=1)) == 0:
        return np.array([[]])
    # Ensure sum(axis=1) has the same number of dimensions as cm to avoid broadcasting issues
    row_sums = cm.sum(axis=1)
    row_sums = np.where(row_sums == 0, 1e-10, row_sums) # Replace 0 with a small number to avoid division by zero
    return cm.astype('float') / row_sums[:, np.newaxis]

# Initialize a list to store results for each model
results_data = []

# --- Logistic Regression Results ---
model_name = "Logistic Regression"

# Ensure lists are not empty before calculating mean/std
if accuracy_lr_scores and f1_lr_scores and confusion_matrices_lr:
    avg_accuracy = np.mean(accuracy_lr_scores)
    std_accuracy = np.std(accuracy_lr_scores)
    avg_f1 = np.mean(f1_lr_scores)
    std_f1 = np.std(f1_lr_scores)
    summed_confusion_matrix_lr = np.sum(confusion_matrices_lr, axis=0)
    normalized_confusion_matrix_lr = normalize_confusion_matrix(summed_confusion_matrix_lr).round(2)
    results_data.append({
        'Model': model_name,
        'Average Accuracy': avg_accuracy,
        'Std Accuracy': std_accuracy,
        'Average F1-score (macro)': avg_f1,
        'Std F1-score (macro)': std_f1,
        'Summed Confusion Matrix': str(summed_confusion_matrix_lr.tolist()),
        'Normalized Summed Confusion Matrix': str(normalized_confusion_matrix_lr.tolist())
    })
else:
    print(f"Skipping {model_name} results as score lists are empty.")

# --- Random Forest Results ---
model_name = "Random Forest"
if accuracy_rf_scores and f1_rf_scores and confusion_matrices_rf:
    avg_accuracy = np.mean(accuracy_rf_scores)
    std_accuracy = np.std(accuracy_rf_scores)
    avg_f1 = np.mean(f1_rf_scores)
    std_f1 = np.std(f1_rf_scores)
    summed_confusion_matrix_rf = np.sum(confusion_matrices_rf, axis=0)
    normalized_confusion_matrix_rf = normalize_confusion_matrix(summed_confusion_matrix_rf).round(2)
    results_data.append({
        'Model': model_name,
        'Average Accuracy': avg_accuracy,
        'Std Accuracy': std_accuracy,
        'Average F1-score (macro)': avg_f1,
        'Std F1-score (macro)': std_f1,
        'Summed Confusion Matrix': str(summed_confusion_matrix_rf.tolist()),
        'Normalized Summed Confusion Matrix': str(normalized_confusion_matrix_rf.tolist())
    })
else:
    print(f"Skipping {model_name} results as score lists are empty.")

# --- XGBoost Results ---
model_name = "XGBoost"
if accuracy_xgb_scores and f1_xgb_scores and confusion_matrices_xgb:
    avg_accuracy = np.mean(accuracy_xgb_scores)
    std_accuracy = np.std(accuracy_xgb_scores)
    avg_f1 = np.mean(f1_xgb_scores)
    std_f1 = np.std(f1_xgb_scores)
    summed_confusion_matrix_xgb = np.sum(confusion_matrices_xgb, axis=0)
    normalized_confusion_matrix_xgb = normalize_confusion_matrix(summed_confusion_matrix_xgb).round(2)
    results_data.append({
        'Model': model_name,
        'Average Accuracy': avg_accuracy,
        'Std Accuracy': std_accuracy,
        'Average F1-score (macro)': avg_f1,
        'Std F1-score (macro)': std_f1,
        'Summed Confusion Matrix': str(summed_confusion_matrix_xgb.tolist()),
        'Normalized Summed Confusion Matrix': str(normalized_confusion_matrix_xgb.tolist())
    })
else:
    print(f"Skipping {model_name} results as score lists are empty.")

# Create a Pandas DataFrame from the results
if results_data:
    df_results = pd.DataFrame(results_data)

    # Save the DataFrame to a CSV file
    csv_filename = "model_performance_summary.csv"
    df_results.to_csv(csv_filename, index=False)

    print(f"Model performance summary saved to {csv_filename}")
    print("\n--- Model Performance Summary ---")
    print(df_results.to_string())
else:
    print("No results to display or save for traditional ML models.")

Model performance summary saved to model_performance_summary.csv

--- Model Performance Summary ---
                 Model  Average Accuracy  Std Accuracy  Average F1-score (macro)  Std F1-score (macro)                            Summed Confusion Matrix                           Normalized Summed Confusion Matrix
0  Logistic Regression          0.868056      0.008793                  0.834433              0.010727  [[1259, 22, 15], [24, 1088, 184], [15, 158, 403]]  [[0.97, 0.02, 0.01], [0.02, 0.84, 0.14], [0.03, 0.27, 0.7]]
1        Random Forest          0.875947      0.008917                  0.815588              0.014334       [[1296, 0, 0], [1, 1199, 96], [0, 296, 280]]      [[1.0, 0.0, 0.0], [0.0, 0.93, 0.07], [0.0, 0.51, 0.49]]
2              XGBoost          0.974116      0.004971                  0.965750              0.006688        [[1294, 2, 0], [0, 1264, 32], [0, 48, 528]]      [[1.0, 0.0, 0.0], [0.0, 0.98, 0.02], [0.0, 0.08, 0.92]]


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, Conv1D, BatchNormalization, GlobalAveragePooling2D,GlobalMaxPooling2D, GlobalAveragePooling1D,GlobalMaxPooling1D
from tensorflow.keras.layers import Dense, Dropout, Concatenate
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import StratifiedKFold # Added StratifiedKFold
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import GaussianNoise
from tensorflow.keras.activations import gelu
from tensorflow.keras.layers import Reshape, TimeDistributed, LSTM, GRU
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, EarlyStopping # Moved imports up
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score # Added imports
from tensorflow.keras.losses import CategoricalCrossentropy # Ensure this is imported for model compilation

# Normalize matGPR
matGPR_reshaped = matGPR.reshape(-1, matGPR.shape[-1])
scaler_GPR = StandardScaler()
matGPR_scaled = scaler_GPR.fit_transform(matGPR_reshaped).reshape(matGPR.shape)

# Normalize matMDall
matMDall_reshaped = matMDall.reshape(-1, matMDall.shape[-1])
scaler_MD = StandardScaler()
matMDall_scaled = scaler_MD.fit_transform(matMDall_reshaped).reshape(matMDall.shape)
# Removed: matMDall_scaled=matMDall_scaled[:,:,:,None] as it's for a different model input shape

# One-hot encode labels (assumed to be integers 0, 1, 2)
labels_cat = to_categorical(labels, num_classes=3)

# Define model parameters based on the last active definitions in the original cell
coef2=5
noConv=32
coef1=7
lr=0.001
dropout=0.3

# First Input: GPR branch
input_gpr = Input(shape=(11, 7, 512), name='GPR_input')
x1 = Conv2D(128*coef1, (3, 3), activation='relu', padding='same')(input_gpr)
x1 = BatchNormalization()(x1)
x1 = Conv2D(64*coef1, (3, 3), activation='relu', padding='same')(x1)
x1 = BatchNormalization()(x1)
x1 = Conv2D(32*coef1, (3, 3), activation='swish', padding='same')(x1)
x1 = BatchNormalization()(x1)
x1 = GlobalAveragePooling2D()(x1)

# Second Input: MDall branch
input_md = Input(shape=(11, 10), name='MD_input') # Matches matMDall_scaled shape
x2 = Conv1D(noConv*coef2, 3, activation='relu', padding='same')(input_md)
x2 = BatchNormalization()(x2)
x2 = Conv1D(noConv//2*coef2, 3, activation='relu', padding='same')(x2)
x2 = BatchNormalization()(x2)
x2 = Conv1D(noConv//4*coef2, 3, activation='swish', padding='same')(x2)
x2 = BatchNormalization()(x2)
x21 = GlobalAveragePooling1D()(x2)
x2 = GlobalMaxPooling1D()(x2)
x2 = Concatenate()([x2, x21])

merged = Concatenate()([x1, x2]) # This model uses only 2 inputs
merged = Dense(256, activation='relu')(merged)
merged = Dense(128, activation='relu')(merged)
merged = Dense(64, activation='relu')(merged)
merged = Dense(32, activation='selu')(merged)
merged = Dropout(dropout)(merged)
output = Dense(3, activation='softmax')(merged)

# Define callbacks for training within the k-fold loop
# Learning rate scheduler
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.85,
    patience=25,
    min_lr=1e-6,
    verbose=1
)

# Early stopping to prevent overfitting
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=60,
    restore_best_weights=True,
    verbose=1
)

# Initialize StratifiedKFold
# skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Lists to store metrics for each fold
accuracy_scores = []
f1_scores = []
confusion_matrices = []

# Use original integer labels for stratification
y_original = labels

for fold, (train_index, val_index) in enumerate(skf.split(matGPR_scaled, y_original)):
    print(f"--- Fold {fold+1}/{skf.n_splits} ---")

    # Split data for the current fold
    X1_train, X1_val = matGPR_scaled[train_index], matGPR_scaled[val_index]
    X2_train, X2_val = matMDall_scaled[train_index], matMDall_scaled[val_index]
    y_train_cat, y_val_cat = labels_cat[train_index], labels_cat[val_index]

    # Recreate and compile the model for each fold to ensure fresh weights
    model = Model(inputs=[input_gpr, input_md], outputs=output)
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss=CategoricalCrossentropy(label_smoothing=0.05),
        metrics=['accuracy']
    )

    # Model checkpoint - saving with fold number
    current_fold_checkpoint = ModelCheckpoint(
        filepath=f'best_model_fold_{fold+1}.keras', # Modified to save per-fold
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=False,
        mode='max',
        verbose=0 # Suppress verbose for checkpoint for cleaner output
    )

    # Train the model
    history = model.fit(
        [X1_train, X2_train], y_train_cat,
        validation_data=([X1_val, X2_val], y_val_cat),
        epochs=350, # Using original epochs
        batch_size=64, # Using original batch_size
        callbacks=[current_fold_checkpoint, lr_scheduler, early_stop], # Adding early_stop
        verbose=1 # Suppress verbose output for cleaner k-fold log
    )

    # Load the best model from the current fold's training
    best_model = tf.keras.models.load_model(f'best_model_fold_{fold+1}.keras')

    # Make predictions on the validation set of the current fold
    yyy_val = best_model.predict([X1_val, X2_val])

    # Convert one-hot encoded y_val to class labels
    y_true_labels = np.argmax(y_val_cat, axis=1);
    # Convert predicted probabilities yyy_val to class labels
    y_pred_labels = np.argmax(yyy_val, axis=1);

    # Calculate and store metrics
    accuracy_scores.append(accuracy_score(y_true_labels, y_pred_labels))
    f1_scores.append(f1_score(y_true_labels, y_pred_labels, average='macro'))
    confusion_matrices.append(confusion_matrix(y_true_labels, y_pred_labels))

# Print aggregated results
print("\n--- Cross-Validation Results ---")
print(f"Average Accuracy: {np.mean(accuracy_scores):.4f} (+/- {np.std(accuracy_scores):.4f})")
print(f"Average F1-score (macro): {np.mean(f1_scores):.4f} (+/- {np.std(f1_scores):.4f})")
print("Sum of Confusion Matrices across all folds:")
print(np.sum(confusion_matrices, axis=0))

# For detailed report, we can use the last fold's predictions as a representative example
print(f"\nClassification Report for the last fold (Fold {fold+1}):")
print(classification_report(y_true_labels, y_pred_labels, digits=4))

--- Fold 1/3 ---
Epoch 1/350
33/33 ━━━━━━━━━━━━━━━━━━━━ 24s 233ms/step - accuracy: 0.5072 - loss: 0.9918 - val_accuracy: 0.3324 - val_loss: 3.3049 - learning_rate: 0.0010
Epoch 2/350
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 78ms/step - accuracy: 0.5308 - loss: 0.9483 - val_accuracy: 0.3589 - val_loss: 1.6696 - learning_rate: 0.0010
Epoch 3/350
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 76ms/step - accuracy: 0.5499 - loss: 0.8945 - val_accuracy: 0.5303 - val_loss: 1.0642 - learning_rate: 0.0010
Epoch 4/350
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 78ms/step - accuracy: 0.6466 - loss: 0.7609 - val_accuracy: 0.6335 - val_loss: 0.9106 - learning_rate: 0.0010
Epoch 5/350
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 82ms/step - accuracy: 0.7397 - loss: 0.5982 - val_accuracy: 0.6402 - val_loss: 0.8695 - learning_rate: 0.0010
Epoch 6/350
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 78ms/step - accuracy: 0.7469 - loss: 0.5706 - val_accuracy: 0.6695 - val_loss: 0.7430 - learning_rate: 0.0010
Epoch 7/350
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 77ms/step - accuracy: 0.767

In [ ]:
accuracy_scores

[0.8162878787878788, 0.8125, 0.9375]

In [ ]:
import pandas as pd
import numpy as np

# Initialize a list to store results for the Deep Learning model
dl_results_data = []

# --- Deep Learning Model Results ---
model_name = "Deep Learning Model"
avg_accuracy = np.mean(accuracy_scores)
std_accuracy = np.std(accuracy_scores)
avg_f1 = np.mean(f1_scores)
std_f1 = np.std(f1_scores)
summed_confusion_matrix = np.sum(confusion_matrices, axis=0)
dl_results_data.append({
    'Model': model_name,
    'Average Accuracy': avg_accuracy,
    'Std Accuracy': std_accuracy,
    'Average F1-score (macro)': avg_f1,
    'Std F1-score (macro)': std_f1,
    'Summed Confusion Matrix': str(summed_confusion_matrix.tolist())
})

# Create a Pandas DataFrame from the results
df_dl_results = pd.DataFrame(dl_results_data)

# Save the DataFrame to a CSV file
csv_filename = "deep_learning_model_summary.csv"
df_dl_results.to_csv(csv_filename, index=False)

print(f"Deep Learning model performance summary saved to {csv_filename}")
print("\n--- Deep Learning Model Performance Summary ---")
print(df_dl_results.to_string())

Deep Learning model performance summary saved to deep_learning_model_summary.csv

--- Deep Learning Model Performance Summary ---
                 Model  Average Accuracy  Std Accuracy  Average F1-score (macro)  Std F1-score (macro)                          Summed Confusion Matrix
0  Deep Learning Model          0.855429      0.058053                  0.804521               0.08563  [[1256, 40, 0], [10, 1108, 178], [0, 230, 346]]


In [ ]:
import pandas as pd
import numpy as np

def normalize_confusion_matrix(cm):
    """Normalizes confusion matrix rows to sum to 1 (percentages)."""
    return cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Initialize a list to store results for each model
results_data = []

# --- Logistic Regression Results ---
model_name = "Logistic Regression"
avg_accuracy = np.mean(accuracy_lr_scores)
std_accuracy = np.std(accuracy_lr_scores)
avg_f1 = np.mean(f1_lr_scores)
std_f1 = np.std(f1_lr_scores)
summed_confusion_matrix_lr = np.sum(confusion_matrices_lr, axis=0)
normalized_confusion_matrix_lr = normalize_confusion_matrix(summed_confusion_matrix_lr).round(2)
results_data.append({
    'Model': model_name,
    'Average Accuracy': avg_accuracy,
    'Std Accuracy': std_accuracy,
    'Average F1-score (macro)': avg_f1,
    'Std F1-score (macro)': std_f1,
    'Summed Confusion Matrix': str(summed_confusion_matrix_lr.tolist()),
    'Normalized Summed Confusion Matrix': str(normalized_confusion_matrix_lr.tolist())
})

# --- Random Forest Results ---
model_name = "Random Forest"
avg_accuracy = np.mean(accuracy_rf_scores)
std_accuracy = np.std(accuracy_rf_scores)
avg_f1 = np.mean(f1_rf_scores)
std_f1 = np.std(f1_rf_scores)
summed_confusion_matrix_rf = np.sum(confusion_matrices_rf, axis=0)
normalized_confusion_matrix_rf = normalize_confusion_matrix(summed_confusion_matrix_rf).round(2)
results_data.append({
    'Model': model_name,
    'Average Accuracy': avg_accuracy,
    'Std Accuracy': std_accuracy,
    'Average F1-score (macro)': avg_f1,
    'Std F1-score (macro)': std_f1,
    'Summed Confusion Matrix': str(summed_confusion_matrix_rf.tolist()),
    'Normalized Summed Confusion Matrix': str(normalized_confusion_matrix_rf.tolist())
})

# --- XGBoost Results ---
model_name = "XGBoost"
avg_accuracy = np.mean(accuracy_xgb_scores)
std_accuracy = np.std(accuracy_xgb_scores)
avg_f1 = np.mean(f1_xgb_scores)
std_f1 = np.std(f1_xgb_scores)
summed_confusion_matrix_xgb = np.sum(confusion_matrices_xgb, axis=0)
normalized_confusion_matrix_xgb = normalize_confusion_matrix(summed_confusion_matrix_xgb).round(2)
results_data.append({
    'Model': model_name,
    'Average Accuracy': avg_accuracy,
    'Std Accuracy': std_accuracy,
    'Average F1-score (macro)': avg_f1,
    'Std F1-score (macro)': std_f1,
    'Summed Confusion Matrix': str(summed_confusion_matrix_xgb.tolist()),
    'Normalized Summed Confusion Matrix': str(normalized_confusion_matrix_xgb.tolist())
})

# Create a Pandas DataFrame from the results
df_results = pd.DataFrame(results_data)

# Save the DataFrame to a CSV file
csv_filename = "model_performance_summary.csv"
df_results.to_csv(csv_filename, index=False)

print(f"Model performance summary saved to {csv_filename}")
print("\n--- Model Performance Summary ---")
print(df_results.to_string())

                 Model  Average Accuracy  Std Accuracy  Average F1-score (macro)  Std F1-score (macro)                           Summed Confusion Matrix
0  Logistic Regression          0.876578      0.013011                  0.846336              0.012742  [[1258, 33, 5], [15, 1100, 181], [11, 146, 419]]
1        Random Forest          0.884470      0.013846                  0.828149              0.019819      [[1296, 0, 0], [3, 1214, 79], [0, 284, 292]]
2              XGBoost          0.988321      0.007436                  0.984528              0.009846       [[1296, 0, 0], [0, 1278, 18], [0, 19, 557]]


In [ ]:
# import numpy as np
# import tensorflow as tf
# from tensorflow.keras.models import Model
# from tensorflow.keras.layers import Input, Conv2D, Conv1D, BatchNormalization, GlobalAveragePooling2D,GlobalMaxPooling2D, GlobalAveragePooling1D,GlobalMaxPooling1D
# from tensorflow.keras.layers import Dense, Dropout, Concatenate
# from tensorflow.keras.optimizers import Adam
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler,MinMaxScaler
# from tensorflow.keras.utils import to_categorical
# from tensorflow.keras.regularizers import l2
# from tensorflow.keras.layers import GaussianNoise
# from tensorflow.keras.activations import gelu
# from tensorflow.keras.layers import Reshape, TimeDistributed, LSTM, GRU
# from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, EarlyStopping

# # Dense(..., kernel_regularizer=l2(1e-4))

# matGPR_reshaped=matGPR.reshape(-1, matGPR.shape[-1])
# scaler_GPR=StandardScaler()
# matGPR_scaled=scaler_GPR.fit_transform(matGPR_reshaped).reshape(matGPR.shape)


# matMDall_reshaped=matMDall.reshape(-1, matMDall.shape[-1])
# scaler_MD=StandardScaler()
# matMDall_scaled=scaler_MD.fit_transform(matMDall_reshaped).reshape(matMDall.shape)
# matMDall_scaled=matMDall_scaled
# matMDall_scaled=matMDall_scaled[:,:,:,None]
# labels_cat=to_categorical(labels, num_classes=3)

# scaler_Combined=StandardScaler()
# combined_array_scaled=scaler_Combined.fit_transform(combined_array)

# X1_train, X1_val, X2_train, X2_val,X3_train, X3_val, y_train, y_val=train_test_split(
#  matGPR_scaled, matMDall_scaled,combined_array_scaled, labels_cat, test_size=0.5, random_state=42, stratify=labels
# )


# coef2=4
# noConv=8
# coef1=2
# lr=0.002
# dropout=0.3
# dropout2=0.5

# input_gpr=Input(shape=(11,7,512), name='GPR_input')
# x0=Reshape((77,512))(input_gpr)
# x0=GRU(64, return_sequences=True)(x0)
# # x0=BatchNormalization()(x0)
# x0=Dropout(dropout2)(x0)
# x0=GlobalMaxPooling1D()(x0)


# # x1=GaussianNoise(0.1)(input_gpr)
# x1=Conv2D(128*coef1,(3, 3), activation='relu', padding='same')(input_gpr)
# x1=BatchNormalization()(x1)
# x1=Dropout(dropout2)(x1)
# x1=Conv2D(64*coef1,(3, 3), activation='relu', padding='same')(x1)
# x1=BatchNormalization()(x1)
# x1=Dropout(dropout2)(x1)
# x1=Conv2D(32*coef1, (3, 3), activation='swish', padding='same')(x1)
# x1=BatchNormalization()(x1)
# x1=GlobalAveragePooling2D()(x1)

# input_md=Input(shape=(11, 10,1), name='MD_input')
# # x2=GaussianNoise(0.1)(input_md)
# x2=Conv2D(noConv*coef2, 3, activation='relu', padding='same')(input_md)


# x2=BatchNormalization()(x2)
# x2=Dropout(dropout2)(x2)
# x2=Conv2D(noConv//2*coef2, 3, activation='relu', padding='same')(x2)
# x2=BatchNormalization()(x2)
# x2=Dropout(dropout2)(x2)
# x2=Conv2D(noConv//4*coef2, 3, activation='swish', padding='same')(x2)
# x2=BatchNormalization()(x2)
# x21=GlobalAveragePooling2D()(x2)
# x2=GlobalMaxPooling2D()(x2)
# x2=Concatenate()([x2,x21])

# input_features=Input(shape=(859,), name='feature_input')
# # x3=GaussianNoise(0.1)(input_features)
# x3=Dense(256,activation='relu')(input_features)
# x3=BatchNormalization()(x3)
# x3=Dropout(dropout2)(x3)
# x3=Dense(128,activation='relu')(x3)
# x3=BatchNormalization()(x3)
# x3=Dropout(dropout2)(x3)
# x3=Dense(8,activation='swish')(x3)
# x3=BatchNormalization()(x3)

# merged=Concatenate()([x1, x2, x3,x0])
# merged=Dense(128, activation='relu')(merged)
# # merged=BatchNormalization()(merged)
# merged=Dropout(dropout2-.1)(merged)
# merged=Dense(64, activation='relu')(merged)
# # merged=BatchNormalization()(merged)
# merged=Dropout(dropout2-0.1)(merged)
# merged=Dense(32, activation='swish')(merged)
# # merged=BatchNormalization()(merged)
# merged=Dropout(dropout2-0.1)(merged)
# output=Dense(3,activation='softmax')(merged)

# lr_scheduler=ReduceLROnPlateau(
#  monitor='val_loss',
#  factor=0.85,
#  patience=25,
#  min_lr=1e-6,
#  verbose=1
# )


# checkpoint=ModelCheckpoint(
#  filepath='best_model.keras',
#  monitor='val_accuracy',
#  save_best_only=True,
#  save_weights_only=False,
#  mode='max',
#  verbose=1
# )

# model=Model(inputs=[input_gpr, input_md, input_features], outputs=output)
# model.compile(optimizer=Adam(learning_rate=lr), loss='categorical_crossentropy', metrics=['accuracy'])

# from tensorflow.keras.losses import CategoricalCrossentropy

# model.compile(
#  optimizer=Adam(learning_rate=lr),
#  loss=CategoricalCrossentropy(label_smoothing=0.05),
#  metrics=['accuracy']
# )

# # Training with all callbacks
# history=model.fit(
#  [X1_train, X2_train, X3_train], y_train,
#  validation_data=([X1_val, X2_val, X3_val], y_val),
#  epochs=230,
#  batch_size=128,
#  callbacks=[checkpoint, lr_scheduler]  # Add all callbacks here
# )

# # Load the best model
# model = tf.keras.models.load_model('best_model.keras')

# yyy=model.predict([X1_val, X2_val, X3_val])

# # Convert one-hot encoded y_val to class labels
# y_true_labels = np.argmax(y_val, axis=1)
# # Convert predicted probabilities yyy to class labels
# y_pred_labels = np.argmax(yyy, axis=1)

# f1_per_class_DN = f1_score(y_true_labels, y_pred_labels, average='macro')
# confusion_matrix_DN = confusion_matrix(y_true_labels, y_pred_labels)
# # Display results
# #print("MD Accuracy with Gradient Boosting:", accuracy_GB)
# print("MD F1 score with DN:", f1_per_class_DN)
# print("MD Confusion Matrix with DN:")
# print(confusion_matrix_DN)
# print(classification_report(y_true_labels, y_pred_labels, digits=4))

Epoch 1/230
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.3657 - loss: 1.5091
Epoch 1: val_accuracy improved from -inf to 0.39331, saving model to best_model.keras
13/13 ━━━━━━━━━━━━━━━━━━━━ 14s 300ms/step - accuracy: 0.3691 - loss: 1.4912 - val_accuracy: 0.3933 - val_loss: 5.3515 - learning_rate: 0.0020
Epoch 2/230
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.4605 - loss: 1.0189
Epoch 2: val_accuracy improved from 0.39331 to 0.44318, saving model to best_model.keras
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 94ms/step - accuracy: 0.4622 - loss: 1.0173 - val_accuracy: 0.4432 - val_loss: 2.7267 - learning_rate: 0.0020
Epoch 3/230
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.5114 - loss: 0.9639
Epoch 3: val_accuracy did not improve from 0.44318
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 92ms/step - accuracy: 0.5137 - loss: 0.9620 - val_accuracy: 0.4312 - val_loss: 2.2732 - learning_rate: 0.0020
Epoch 4/230
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.5681 - loss: 0.8957
Epoch 4:

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, Conv1D, BatchNormalization, GlobalAveragePooling2D,GlobalMaxPooling2D, GlobalAveragePooling1D,GlobalMaxPooling1D
from tensorflow.keras.layers import Dense, Dropout, Concatenate
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import StratifiedKFold # Added StratifiedKFold
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import GaussianNoise
from tensorflow.keras.activations import gelu
from tensorflow.keras.layers import Reshape, TimeDistributed, LSTM, GRU
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, EarlyStopping # Moved imports up
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score # Added imports
from tensorflow.keras.losses import CategoricalCrossentropy # Ensure this is imported for model compilation

# Normalize matGPR
matGPR_reshaped = matGPR.reshape(-1, matGPR.shape[-1])
scaler_GPR = StandardScaler()
matGPR_scaled = scaler_GPR.fit_transform(matGPR_reshaped).reshape(matGPR.shape)

# Normalize matMDall
matMDall_reshaped = matMDall.reshape(-1, matMDall.shape[-1])
scaler_MD = StandardScaler()
matMDall_scaled = scaler_MD.fit_transform(matMDall_reshaped).reshape(matMDall.shape)
matMDall_scaled=matMDall_scaled[:,:,:,None] # Ensure 4D for Conv2D

# One-hot encode labels (assumed to be integers 0, 1, 2)
labels_cat = to_categorical(labels, num_classes=3)

# Normalize combined_array for feature input
scaler_Combined = StandardScaler()
combined_array_scaled = scaler_Combined.fit_transform(combined_array)


# Define model parameters based on the last active definitions in VKyYjcM4BV9g
coef2=4
noConv=32
coef1=6
lr=0.001
dropout=0.8


# First Input: GPR branch
input_gpr = Input(shape=(11, 7, 512), name='GPR_input')
x1 = Conv2D(128*coef1, (3, 3), activation='relu', padding='same')(input_gpr)
x1 = BatchNormalization()(x1)
x1 = Conv2D(64*coef1, (3, 3), activation='relu', padding='same')(x1)
x1 = BatchNormalization()(x1)
x1 = Conv2D(32*coef1, (3, 3), activation='selu', padding='same')(x1)
x1 = BatchNormalization()(x1)
x1 = GlobalAveragePooling2D()(x1)

# Second Input: MDall branch - using Conv2D, so input shape must be 4D
input_md = Input(shape=(11, 10, 1), name='MD_input') # Changed to 4D for Conv2D
x2 = Conv2D(noConv*coef2, 3, activation='relu', padding='same')(input_md) # Changed to Conv2D
x2 = BatchNormalization()(x2)
x2 = Conv2D(noConv//2*coef2, 3, activation='relu', padding='same')(x2) # Changed to Conv2D
x2 = BatchNormalization()(x2)
x2 = Conv2D(noConv//4*coef2, 3, activation='selu', padding='same')(x2) # Changed to Conv2D
x2 = BatchNormalization()(x2)
x21 = GlobalAveragePooling2D()(x2) # Changed to 2D
x2 = GlobalMaxPooling2D()(x2) # Changed to 2D
x2 = Concatenate()([x2, x21])

# Third Input: Features branch
input_features = Input(shape=(859,), name='feature_input')
x3 = Dense(256, activation='relu')(input_features)
x3 = BatchNormalization()(x3)
x3 = Dense(128, activation='relu')(x3)
x3 = BatchNormalization()(x3)
x3 = Dense(16, activation='selu')(x3)
x3 = BatchNormalization()(x3)

# Merged layers
merged = Concatenate()([x1, x2, x3]) # Using all three inputs
merged = Dense(256, activation='relu')(merged)
# merged = Dropout(dropout*.1)(merged) # Removed comments for consistency if dropout was intended
merged = Dense(128, activation='relu')(merged)
# merged = Dropout(dropout*.1)(merged)
merged = Dense(64, activation='relu')(merged)
# merged = Dropout(dropout*.1)(merged)
merged = Dense(32, activation='selu')(merged)
# merged = Dropout(dropout*.1)(merged)
# merged = Dense(21, activation='relu')(merged)
merged = Dropout(dropout)(merged) # Using the defined dropout
output = Dense(3, activation='softmax')(merged)

# Define callbacks for training within the k-fold loop
# Learning rate scheduler
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.85,
    patience=25,
    min_lr=1e-6,
    verbose=0 # Suppress verbose for scheduler output for cleaner k-fold log
)

# Early stopping to prevent overfitting
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=60,
    restore_best_weights=True,
    verbose=0 # Suppress verbose for early stopping for cleaner k-fold log
)

# Initialize StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Lists to store metrics for each fold
accuracy_scores = []
f1_scores = []
confusion_matrices = []

# Use original integer labels for stratification
y_original = labels

for fold, (train_index, val_index) in enumerate(skf.split(matGPR_scaled, y_original)): # Only one X for split, but apply indices to all
    print(f"--- Fold {fold+1}/{skf.n_splits} ---")

    # Split data for the current fold
    X1_train, X1_val = matGPR_scaled[train_index], matGPR_scaled[val_index]
    X2_train, X2_val = matMDall_scaled[train_index], matMDall_scaled[val_index]
    X3_train, X3_val = combined_array_scaled[train_index], combined_array_scaled[val_index] # Split features
    y_train_cat, y_val_cat = labels_cat[train_index], labels_cat[val_index]

    # Recreate and compile the model for each fold to ensure fresh weights
    model = Model(inputs=[input_gpr, input_md, input_features], outputs=output) # Added input_features
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss=CategoricalCrossentropy(label_smoothing=0.05),
        metrics=['accuracy']
    )

    # Model checkpoint - saving with fold number
    current_fold_checkpoint = ModelCheckpoint(
        filepath=f'best_model_fold_{fold+1}.keras',
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=False,
        mode='max',
        verbose=0
    )

    # Train the model
    history = model.fit(
        [X1_train, X2_train, X3_train], y_train_cat, # Added X3_train
        validation_data=([X1_val, X2_val, X3_val], y_val_cat), # Added X3_val
        epochs=200, # Changed from 350 to 200, consistent with VKyYjcM4BV9g
        batch_size=16, # Changed from 64 to 16, consistent with VKyYjcM4BV9g
        callbacks=[current_fold_checkpoint, lr_scheduler, early_stop],
        verbose=0
    )

    # Load the best model from the current fold's training
    best_model = tf.keras.models.load_model(f'best_model_fold_{fold+1}.keras')

    # Make predictions on the validation set of the current fold
    yyy_val = best_model.predict([X1_val, X2_val, X3_val]) # Added X3_val

    # Convert one-hot encoded y_val to class labels
    y_true_labels = np.argmax(y_val_cat, axis=1)
    # Convert predicted probabilities yyy_val to class labels
    y_pred_labels = np.argmax(yyy_val, axis=1)

    # Calculate and store metrics
    accuracy_scores.append(accuracy_score(y_true_labels, y_pred_labels))
    f1_scores.append(f1_score(y_true_labels, y_pred_labels, average='macro'))
    confusion_matrices.append(confusion_matrix(y_true_labels, y_pred_labels))

# Print aggregated results
print("\n--- Cross-Validation Results ---")
print(f"Average Accuracy: {np.mean(accuracy_scores):.4f} (+/- {np.std(accuracy_scores):.4f})")
print(f"Average F1-score (macro): {np.mean(f1_scores):.4f} (+/- {np.std(f1_scores):.4f})")
print("Sum of Confusion Matrices across all folds:")
print(np.sum(confusion_matrices, axis=0))

# For detailed report, we can use the last fold's predictions as a representative example
print(f"\nClassification Report for the last fold (Fold {fold+1}):")
print(classification_report(y_true_labels, y_pred_labels, digits=4))

Epoch 1/350
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 271ms/step - accuracy: 0.4679 - loss: 1.0101
Epoch 1: val_accuracy improved from -inf to 0.39905, saving model to best_model1.keras
40/40 ━━━━━━━━━━━━━━━━━━━━ 34s 435ms/step - accuracy: 0.4689 - loss: 1.0085 - val_accuracy: 0.3991 - val_loss: 5.2129
Epoch 2/350
39/40 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.5431 - loss: 0.8884
Epoch 2: val_accuracy did not improve from 0.39905
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 60ms/step - accuracy: 0.5435 - loss: 0.8874 - val_accuracy: 0.3628 - val_loss: 2.0603
Epoch 3/350
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.5791 - loss: 0.8237
Epoch 3: val_accuracy improved from 0.39905 to 0.49211, saving model to best_model1.keras
40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 72ms/step - accuracy: 0.5798 - loss: 0.8222 - val_accuracy: 0.4921 - val_loss: 1.4345
Epoch 4/350
39/40 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.6548 - loss: 0.6427
Epoch 4: val_accuracy did not improve from 0.49211
40/40 ━━━━━━━━━━━━━━━━━━━━

NameError: name 'f1_score' is not defined

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, Conv1D, BatchNormalization, GlobalAveragePooling2D,GlobalMaxPooling2D, GlobalAveragePooling1D,GlobalMaxPooling1D
from tensorflow.keras.layers import Dense, Dropout, Concatenate
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import StratifiedKFold # Added StratifiedKFold
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import GaussianNoise
from tensorflow.keras.activations import gelu
from tensorflow.keras.layers import Reshape, TimeDistributed, LSTM, GRU
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, EarlyStopping # Moved imports up
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score # Added imports
from tensorflow.keras.losses import CategoricalCrossentropy # Ensure this is imported for model compilation

# Normalize matGPR
matGPR_reshaped = matGPR.reshape(-1, matGPR.shape[-1])
scaler_GPR = StandardScaler()
matGPR_scaled = scaler_GPR.fit_transform(matGPR_reshaped).reshape(matGPR.shape)

# Normalize matMDall
matMDall_reshaped = matMDall.reshape(-1, matMDall.shape[-1])
scaler_MD = StandardScaler()
matMDall_scaled = scaler_MD.fit_transform(matMDall_reshaped).reshape(matMDall.shape)
# Removed: matMDall_scaled=matMDall_scaled[:,:,:,None] as it's for a different model input shape

# One-hot encode labels (assumed to be integers 0, 1, 2)
labels_cat = to_categorical(labels, num_classes=3)

# Define model parameters based on the last active definitions in the original cell
coef2=5
noConv=32
coef1=7
lr=0.001
dropout=0.3

# First Input: GPR branch
input_gpr = Input(shape=(11, 7, 512), name='GPR_input')
x1 = Conv2D(128*coef1, (3, 3), activation='relu', padding='same')(input_gpr)
x1 = BatchNormalization()(x1)
x1 = Conv2D(64*coef1, (3, 3), activation='relu', padding='same')(x1)
x1 = BatchNormalization()(x1)
x1 = Conv2D(32*coef1, (3, 3), activation='swish', padding='same')(x1)
x1 = BatchNormalization()(x1)
x1 = GlobalAveragePooling2D()(x1)

# Second Input: MDall branch
input_md = Input(shape=(11, 10), name='MD_input') # Matches matMDall_scaled shape
x2 = Conv1D(noConv*coef2, 3, activation='relu', padding='same')(input_md)
x2 = BatchNormalization()(x2)
x2 = Conv1D(noConv//2*coef2, 3, activation='relu', padding='same')(x2)
x2 = BatchNormalization()(x2)
x2 = Conv1D(noConv//4*coef2, 3, activation='swish', padding='same')(x2)
x2 = BatchNormalization()(x2)
x21 = GlobalAveragePooling1D()(x2)
x2 = GlobalMaxPooling1D()(x2)
x2 = Concatenate()([x2, x21])

merged = Concatenate()([x1, x2]) # This model uses only 2 inputs
merged = Dense(256, activation='relu')(merged)
merged = Dense(128, activation='relu')(merged)
merged = Dense(64, activation='relu')(merged)
merged = Dense(32, activation='selu')(merged)
merged = Dropout(dropout)(merged)
output = Dense(3, activation='softmax')(merged)

# Define callbacks for training within the k-fold loop
# Learning rate scheduler
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.85,
    patience=25,
    min_lr=1e-6,
    verbose=1
)

# Early stopping to prevent overfitting
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=60,
    restore_best_weights=True,
    verbose=1
)

# Initialize StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Lists to store metrics for each fold
accuracy_scores = []
f1_scores = []
confusion_matrices = []

# Use original integer labels for stratification
y_original = labels

for fold, (train_index, val_index) in enumerate(skf.split(matGPR_scaled, y_original)):
    print(f"--- Fold {fold+1}/{skf.n_splits} ---")

    # Split data for the current fold
    X1_train, X1_val = matGPR_scaled[train_index], matGPR_scaled[val_index]
    X2_train, X2_val = matMDall_scaled[train_index], matMDall_scaled[val_index]
    y_train_cat, y_val_cat = labels_cat[train_index], labels_cat[val_index]

    # Recreate and compile the model for each fold to ensure fresh weights
    model = Model(inputs=[input_gpr, input_md], outputs=output)
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss=CategoricalCrossentropy(label_smoothing=0.05),
        metrics=['accuracy']
    )

    # Model checkpoint - saving with fold number
    current_fold_checkpoint = ModelCheckpoint(
        filepath=f'best_model_fold_{fold+1}.keras', # Modified to save per-fold
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=False,
        mode='max',
        verbose=0 # Suppress verbose for checkpoint for cleaner output
    )

    # Train the model
    history = model.fit(
        [X1_train, X2_train], y_train_cat,
        validation_data=([X1_val, X2_val], y_val_cat),
        epochs=350, # Using original epochs
        batch_size=64, # Using original batch_size
        callbacks=[current_fold_checkpoint, lr_scheduler, early_stop], # Adding early_stop
        verbose=0 # Suppress verbose output for cleaner k-fold log
    )

    # Load the best model from the current fold's training
    best_model = tf.keras.models.load_model(f'best_model_fold_{fold+1}.keras')

    # Make predictions on the validation set of the current fold
    yyy_val = best_model.predict([X1_val, X2_val])

    # Convert one-hot encoded y_val to class labels
    y_true_labels = np.argmax(y_val_cat, axis=1);
    # Convert predicted probabilities yyy_val to class labels
    y_pred_labels = np.argmax(yyy_val, axis=1);

    # Calculate and store metrics
    accuracy_scores.append(accuracy_score(y_true_labels, y_pred_labels))
    f1_scores.append(f1_score(y_true_labels, y_pred_labels, average='macro'))
    confusion_matrices.append(confusion_matrix(y_true_labels, y_pred_labels))

# Print aggregated results
print("\n--- Cross-Validation Results ---")
print(f"Average Accuracy: {np.mean(accuracy_scores):.4f} (+/- {np.std(accuracy_scores):.4f})")
print(f"Average F1-score (macro): {np.mean(f1_scores):.4f} (+/- {np.std(f1_scores):.4f})")
print("Sum of Confusion Matrices across all folds:")
print(np.sum(confusion_matrices, axis=0))

# For detailed report, we can use the last fold's predictions as a representative example
print(f"\nClassification Report for the last fold (Fold {fold+1}):")
print(classification_report(y_true_labels, y_pred_labels, digits=4))

20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 121ms/step
MD F1 score with DN: 0.8012969048439612
MD Confusion Matrix with DN:
[[259   1   0]
 [ 19 198  42]
 [  4  36  75]]
              precision    recall  f1-score   support

           0     0.9184    0.9962    0.9557       260
           1     0.8426    0.7645    0.8016       259
           2     0.6410    0.6522    0.6466       115

    accuracy                         0.8391       634
   macro avg     0.8007    0.8043    0.8013       634
weighted avg     0.8371    0.8391    0.8367       634



In [ ]:
matMDall_scaled1=matMDall_scaled[:,:,:,None]

matMDall_scaled1.shape

(3168, 11, 10, 1)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, Conv1D, BatchNormalization, GlobalAveragePooling2D,GlobalMaxPooling2D, GlobalAveragePooling1D,GlobalMaxPooling1D
from tensorflow.keras.layers import Dense, Dropout, Concatenate
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import GaussianNoise
from tensorflow.keras.activations import gelu
from tensorflow.keras.layers import Reshape, TimeDistributed, LSTM, GRU
# Dense(..., kernel_regularizer=l2(1e-4))


# Normalize matGPR
matGPR_reshaped = matGPR.reshape(-1, matGPR.shape[-1])
scaler_GPR = StandardScaler()
matGPR_scaled = scaler_GPR.fit_transform(matGPR_reshaped).reshape(matGPR.shape)

# Normalize matMDall
matMDall_reshaped = matMDall.reshape(-1, matMDall.shape[-1])
scaler_MD = StandardScaler()
matMDall_scaled = scaler_MD.fit_transform(matMDall_reshaped).reshape(matMDall.shape)
matMDall_scaled=matMDall_scaled[:,:,:,None]
labels_cat = to_categorical(labels, num_classes=3)

scaler_Combined = StandardScaler()
combined_array_scaled = scaler_Combined.fit_transform(combined_array)

X1_train, X1_val, X2_train, X2_val,X3_train, X3_val, y_train, y_val = train_test_split(
    matGPR_scaled, matMDall_scaled,combined_array_scaled, labels_cat, test_size=0.5, random_state=42, stratify=labels
)

coef2=4
noConv=32
coef1=6; lr=0.001; dropout=0.8


input_gpr = Input(shape=(11, 7, 512), name='GPR_input')
x1 = Conv2D(128*coef1, (3, 3), activation='relu', padding='same')(input_gpr)
x1 = BatchNormalization()(x1)
x1 = Conv2D(64*coef1, (3, 3), activation='relu', padding='same')(x1)
x1 = BatchNormalization()(x1)
x1 = Conv2D(32*coef1, (3, 3), activation='selu', padding='same')(x1)
x1 = BatchNormalization()(x1)
x1 = GlobalAveragePooling2D()(x1)

input_md = Input(shape=(11, 10,1), name='MD_input')
x2 = Conv2D(noConv*coef2, 3, activation='relu', padding='same')(input_md)
x2 = BatchNormalization()(x2)
x2 = Conv2D(noConv//2*coef2, 3, activation='relu', padding='same')(x2)
x2 = BatchNormalization()(x2)
x2 = Conv2D(noConv//4*coef2, 3, activation='selu', padding='same')(x2)
x2 = BatchNormalization()(x2)
x21 = GlobalAveragePooling2D()(x2)
x2 = GlobalMaxPooling2D()(x2)
x2 = Concatenate()([x2, x21])

input_features = Input(shape=(859,), name='feature_input')
x3 = Dense(256, activation='relu')(input_features)
x3 = BatchNormalization()(x3)
x3 = Dense(128, activation='relu')(x3)
x3 = BatchNormalization()(x3)
x3 = Dense(16, activation='selu')(x3)
x3 = BatchNormalization()(x3)

merged = Concatenate()([x1, x2, x3])
merged = Dense(256, activation='relu')(merged)
# merged = Dropout(dropout*.1)(merged)
merged = Dense(128, activation='relu')(merged)
# merged = Dropout(dropout*.1)(merged)
merged = Dense(64, activation='relu')(merged)
# merged = Dropout(dropout*.1)(merged)
merged = Dense(32, activation='selu')(merged)
# merged = Dropout(dropout*.1)(merged)
# merged = Dense(21, activation='relu')(merged)
merged = Dropout(dropout)(merged)
output = Dense(3, activation='softmax')(merged)






coef2 = 2
noConv = 16
coef1 = 2
lr = 0.002
dropout = 0.3

# GPR branch - add dropout between layers
# input_gpr = Input(shape=(11, 7, 512), name='GPR_input')

# x0 = Reshape((77, 512))(input_gpr)
# x0 = LSTM(32, return_sequences=True)(x0)
# # x0 = BatchNormalization()(x0)
# x0 = Dropout(0.3)(x0)
# x0 = GlobalAveragePooling1D()(x0)

# # Add to input pipelines
# x1 = GaussianNoise(0.1)(input_gpr)
# x1 = Conv2D(128*coef1, (3, 3), activation='relu', padding='same')(x1)
# x1 = BatchNormalization()(x1)
# x1 = Dropout(0.2)(x1)  # Add dropout
# x1 = Conv2D(64*coef1, (3, 3), activation='relu', padding='same')(x1)
# x1 = BatchNormalization()(x1)
# x1 = Dropout(0.2)(x1)  # Add dropout
# x1 = Conv2D(32*coef1, (3, 3), activation='swish', padding='same')(x1)
# x1 = BatchNormalization()(x1)
# x1 = GlobalAveragePooling2D()(x1)

# # MD branch - add dropout
# input_md = Input(shape=(11, 10), name='MD_input')
# x2 = GaussianNoise(0.1)(input_md)
# x2 = Conv1D(noConv*coef2, 3, activation='relu', padding='same')(x2)


# x2 = BatchNormalization()(x2)
# x2 = Dropout(0.2)(x2)
# x2 = Conv1D(noConv//2*coef2, 3, activation='relu', padding='same')(x2)
# x2 = BatchNormalization()(x2)
# x2 = Dropout(0.2)(x2)
# x2 = Conv1D(noConv//4*coef2, 3, activation='swish', padding='same')(x2)
# x2 = BatchNormalization()(x2)
# x21 = GlobalAveragePooling1D()(x2)
# x2 = GlobalMaxPooling1D()(x2)
# x2 = Concatenate()([x2, x21])

# # Feature branch - less aggressive reduction
# input_features = Input(shape=(859,), name='feature_input')
# x3 = GaussianNoise(0.1)(input_features)
# x3 = Dense(256, activation='relu')(x3)
# x3 = BatchNormalization()(x3)
# x3 = Dropout(0.3)(x3)
# x3 = Dense(128, activation='relu')(x3)
# x3 = BatchNormalization()(x3)
# x3 = Dropout(0.3)(x3)
# x3 = Dense(8, activation='swish')(x3)  # Less aggressive: 64 instead of 16
# x3 = BatchNormalization()(x3)

# # Final layers - distribute dropout
# merged = Concatenate()([x1, x2, x3,x0])
# merged = Dense(128, activation='relu')(merged)  # Reduced from 256
# merged = Dropout(0.4)(merged)
# merged = Dense(64, activation='relu')(merged)
# merged = Dropout(0.4)(merged)
# merged = Dense(32, activation='swish')(merged)
# merged = Dropout(0.5)(merged)  # Reduced final dropout
# output = Dense(3, activation='softmax')(merged)


from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, EarlyStopping

# Learning rate scheduler
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.85,
    patience=25,
    min_lr=1e-6,
    verbose=1  # Add this to see when LR changes
)

# Early stopping to prevent overfitting
# early_stop = EarlyStopping(
#     monitor='val_loss',
#     patience=60,  # Wait longer than lr_scheduler patience
#     restore_best_weights=True,
#     verbose=1
# )

# Model checkpoint
checkpoint = ModelCheckpoint(
    filepath='best_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    save_weights_only=False,
    mode='max',
    verbose=1
)

# Build and compile model
model = Model(inputs=[input_gpr, input_md, input_features], outputs=output)
model.compile(optimizer=Adam(learning_rate=lr), loss='categorical_crossentropy', metrics=['accuracy'])

from tensorflow.keras.losses import CategoricalCrossentropy

model.compile(
    optimizer=Adam(learning_rate=lr),
    loss=CategoricalCrossentropy(label_smoothing=0.05),  # Try 0.05 to 0.1
    metrics=['accuracy']
)

# Training with all callbacks
history = model.fit(
    [X1_train, X2_train, X3_train], y_train,
    validation_data=([X1_val, X2_val, X3_val], y_val),
    epochs=200,
    batch_size=16,
    callbacks=[checkpoint, lr_scheduler]  # Add all callbacks here
)

Epoch 1/200
97/99 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.4283 - loss: 1.4919
Epoch 1: val_accuracy improved from -inf to 0.75063, saving model to best_model.keras
99/99 ━━━━━━━━━━━━━━━━━━━━ 20s 58ms/step - accuracy: 0.4307 - loss: 1.4830 - val_accuracy: 0.7506 - val_loss: 0.7073 - learning_rate: 0.0020
Epoch 2/200
97/99 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.6783 - loss: 0.8016
Epoch 2: val_accuracy improved from 0.75063 to 0.76136, saving model to best_model.keras
99/99 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.6793 - loss: 0.8000 - val_accuracy: 0.7614 - val_loss: 0.6202 - learning_rate: 0.0020
Epoch 3/200
97/99 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.7855 - loss: 0.6202
Epoch 3: val_accuracy improved from 0.76136 to 0.77841, saving model to best_model.keras
99/99 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.7854 - loss: 0.6203 - val_accuracy: 0.7784 - val_loss: 0.6048 - learning_rate: 0.0020
Epoch 4/200
97/99 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - acc

# Task
Update the CNN training loop in cell `xJxdd72Oua_G` by adding `ModelCheckpoint` and `EarlyStopping` callbacks. Ensure that the best model weights are loaded after each fold's training for prediction and evaluation.

## Update CNN Training Loop with Callbacks and Best Model Loading

### Subtask:
Modify the CNN training loop in cell 'xJxdd72Oua_G' to include ModelCheckpoint and EarlyStopping callbacks. Ensure that the best model weights are loaded after each fold's training for prediction and evaluation.


**Reasoning**:
The subtask requires modifying the CNN training loop in cell 'xJxdd72Oua_G' to incorporate `ModelCheckpoint` and `EarlyStopping` callbacks, and then loading the best model for evaluation after each fold. I will apply all the requested changes in a single code block for clarity and to ensure correct integration.



In [ ]:
import scipy.io
import numpy as np

gpr59=scipy.io.loadmat('./GPR_m59Data.mat')['GPR_m59Data']
gprSoil = scipy.io.loadmat('./GPR_SoilData.mat')['GPR_SoilData']
gprClutter=scipy.io.loadmat('./GPR_ClutterData.mat')['GPR_ClutterData']
matGPR=np.concatenate((gpr59, gprSoil, gprClutter), axis=3)

print(gpr59.shape)
print(gprSoil.shape)
print(gprClutter.shape)


matGPR=np.concatenate((gpr59, gprSoil, gprClutter), axis=3)
print('Shape of the loaded data before rotation is', matGPR.shape)


matGPR = np.rot90(matGPR,1,(0,3))
matGPR = np.rot90(matGPR,1,(1,3))
matGPR = np.rot90(matGPR,1,(2,3))
matGPR = np.fliplr(matGPR)
# Print to check
print('Shape of the loaded data after rotation is', matGPR.shape)


# 0 is for class Landmine (danger present)
# 1 is for class Soil
# 2 is for class Clutter
labels = np.zeros(264)
labels[108:216]=1
labels[216:]=2


def getPulseDerivative(data):
     dataD=data.copy()
     for t in range(1,data.shape[3]):
          dataD[:,:,:,t]=data[:,:,:,t]-data[:,:,:,t-1]
     dataD=dataD[:,:,:,1:]
     return dataD



def getPulseFeatures(data,func):
     features = []
     for n in range(data.shape[0]):
          for m in range(data.shape[1]):
               features.append(func(data[n,m,:]))
     features = np.array(features)
     return features



def getHistogramData(data,bins=4):
     hist = np.histogram(data,bins)
     hist = hist[0]
     return hist



def getAllFeatures(data):
    FeaturesData = np.empty(shape=[0,632])              # features of a version of the dataset
    dataD = getPulseDerivative(data)
    for r in range(data.shape[0]):
        features = []                               # features of each record (measurement) of size 11x7x512
        hist = getHistogramData(data[r,:,:,:])
        histD = getHistogramData(dataD[r,:,:,:])
        maxAll = np.max(data[r,:,:,:])
        minAll = np.min(data[r,:,:,:])
        meanAll = np.mean(data[r,:,:,:])
        stdAll = np.std(data[r,:,:,:])
        maxAllD = np.max(dataD[r,:,:,:])
        minAllD = np.min(dataD[r,:,:,:])
        meanAllD = np.mean(dataD[r,:,:,:])
        stdAllD = np.std(dataD[r,:,:,:])
        features = [maxAll,maxAllD,minAll,minAllD,meanAll,meanAllD,stdAll,stdAllD]
        features = np.concatenate((features, hist, histD))
        features = np.concatenate((features,getPulseFeatures(data[r,:,:,:], np.max)))
        features = np.concatenate((features,getPulseFeatures(data[r,:,:,:], np.min)))
        features = np.concatenate((features,getPulseFeatures(data[r,:,:,:], np.mean)))
        features = np.concatenate((features,getPulseFeatures(data[r,:,:,:], np.std)))
        features = np.concatenate((features,getPulseFeatures(dataD[r,:,:,:], np.max)))
        features = np.concatenate((features,getPulseFeatures(dataD[r,:,:,:], np.min)))
        features = np.concatenate((features,getPulseFeatures(dataD[r,:,:,:], np.mean)))
        features = np.concatenate((features,getPulseFeatures(dataD[r,:,:,:], np.std)))
        features = features.reshape(1,-1)
        FeaturesData = np.concatenate((FeaturesData,features),axis=0)

    return FeaturesData




# Augment the data using flipping
matGPR1 = np.flip(matGPR,1)
matGPR2 = np.flip(matGPR,2)
matGPR12 = np.flip(np.flip(matGPR,1),2)
matGPR = np.concatenate((matGPR,matGPR1,matGPR2,matGPR12), axis=0)
# New shape should be 1056*11*7*512
print('Shape of the data after flipping augmentation is', matGPR.shape)

AllFeatures = getAllFeatures(matGPR)
print('Shape of the features data is', AllFeatures.shape)
data_set = AllFeatures
labels = np.concatenate((labels, labels, labels, labels), axis=0)




from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

X = data_set.copy()
y = labels.copy()
print("X Shape:", X.shape)
print("y Shape:", y.shape)


skf = StratifiedKFold(n_splits=3,shuffle=True,random_state=42)



print('\n\nLogistic Regression')
from sklearn.linear_model import LogisticRegression
modelLR = LogisticRegression(max_iter=1000000)
all_y_test, all_y_pred = [], []
accuracy, precision, recall, f1 = [], [], [], []
for fold, (train_index, test_index) in enumerate(skf.split(X, y)):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    modelLR.fit(X_train, y_train)
    y_pred = modelLR.predict(X_test)
    all_y_test.extend(y_test)
    all_y_pred.extend(y_pred)
    accuracy.append(accuracy_score(y_test, y_pred))
    precision.append(precision_score(y_test, y_pred, average='macro'))
    recall.append(recall_score(y_test, y_pred, average='macro'))
    f1.append(f1_score(y_test, y_pred, average='macro'))
# Display results
print("GPR Accuracy with Logistic Regression:", np.mean(accuracy), "+-", np.std(accuracy))
print("GPR Precision with Logistic Regression:", np.mean(precision), "+-", np.std(precision))
print("GPR Recall with Logistic Regression:", np.mean(recall), "+-", np.std(recall))
print("GPR F1 score with Logistic Regression:", np.mean(f1), "+-", np.std(f1))
print("GPR Overall Confusion Matrix with Logistic Regression:")
print(confusion_matrix(all_y_test, all_y_pred))


print('\n\nRandom Forest')
from sklearn.ensemble import RandomForestClassifier
modelRF = RandomForestClassifier(n_estimators=100, random_state=42)
all_y_test, all_y_pred = [], []
accuracy, precision, recall, f1 = [], [], [], []
for fold, (train_index, test_index) in enumerate(skf.split(X, y)):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    modelRF.fit(X_train, y_train)
    y_pred = modelRF.predict(X_test)
    all_y_test.extend(y_test)
    all_y_pred.extend(y_pred)
    accuracy.append(accuracy_score(y_test, y_pred))
    precision.append(precision_score(y_test, y_pred, average='macro'))
    recall.append(recall_score(y_test, y_pred, average='macro'))
    f1.append(f1_score(y_test, y_pred, average='macro'))
# Display results
print("GPR Accuracy with Random Forest:", np.mean(accuracy), "+-", np.std(accuracy))
print("GPR Precision with Random Forest:", np.mean(precision), "+-", np.std(precision))
print("GPR Recall with Random Forest:", np.mean(recall), "+-", np.std(recall))
print("GPR F1 score with Random Forest:", np.mean(f1), "+-", np.std(f1))
print("GPR Overall Confusion Matrix with Random Forest:")
print(confusion_matrix(all_y_test, all_y_pred))



print('\n\nSVC')
from sklearn.svm import SVC
modelSVC = SVC(kernel='linear')
all_y_test, all_y_pred = [], []
accuracy, precision, recall, f1 = [], [], [], []
for fold, (train_index, test_index) in enumerate(skf.split(X, y)):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    modelSVC.fit(X_train, y_train)
    y_pred = modelSVC.predict(X_test)
    all_y_test.extend(y_test)
    all_y_pred.extend(y_pred)
    accuracy.append(accuracy_score(y_test, y_pred))
    precision.append(precision_score(y_test, y_pred, average='macro'))
    recall.append(recall_score(y_test, y_pred, average='macro'))
    f1.append(f1_score(y_test, y_pred, average='macro'))
# Display results
print("GPR Accuracy with SVC:", np.mean(accuracy), "+-", np.std(accuracy))
print("GPR Precision with SVC:", np.mean(precision), "+-", np.std(precision))
print("GPR Recall with SVC:", np.mean(recall), "+-", np.std(recall))
print("GPR F1 score with SVC:", np.mean(f1), "+-", np.std(f1))
print("GPR Overall Confusion Matrix with SVC:")
print(confusion_matrix(all_y_test, all_y_pred))



print('\n\nKNN')
from sklearn.neighbors import KNeighborsClassifier
modelKNN = KNeighborsClassifier(n_neighbors=5)
all_y_test, all_y_pred = [], []
accuracy, precision, recall, f1 = [], [], [], []
for fold, (train_index, test_index) in enumerate(skf.split(X, y)):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    modelKNN.fit(X_train, y_train)
    y_pred = modelKNN.predict(X_test)
    all_y_test.extend(y_test)
    all_y_pred.extend(y_pred)
    accuracy.append(accuracy_score(y_test, y_pred))
    precision.append(precision_score(y_test, y_pred, average='macro',zero_division=0))
    recall.append(recall_score(y_test, y_pred, average='macro'))
    f1.append(f1_score(y_test, y_pred, average='macro'))
# Display results
print("GPR Accuracy with KNN:", np.mean(accuracy), "+-", np.std(accuracy))
print("GPR Precision with KNN:", np.mean(precision), "+-", np.std(precision))
print("GPR Recall with KNN:", np.mean(recall), "+-", np.std(recall))
print("GPR F1 score with KNN:", np.mean(f1), "+-", np.std(f1))
print("GPR Overall Confusion Matrix with KNN:")
print(confusion_matrix(all_y_test, all_y_pred))




print('\n\nGradient Boosting')
from sklearn.ensemble import GradientBoostingClassifier
modelGB = GradientBoostingClassifier(n_estimators=100, random_state=42)
all_y_test, all_y_pred = [], []
accuracy, precision, recall, f1 = [], [], [], []
for fold, (train_index, test_index) in enumerate(skf.split(X, y)):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    modelGB.fit(X_train, y_train)
    y_pred = modelGB.predict(X_test)
    all_y_test.extend(y_test)
    all_y_pred.extend(y_pred)
    accuracy.append(accuracy_score(y_test, y_pred))
    precision.append(precision_score(y_test, y_pred, average='macro'))
    recall.append(recall_score(y_test, y_pred, average='macro'))
    f1.append(f1_score(y_test, y_pred, average='macro'))
# Display results
print("GPR Accuracy with Gradient Boosting:", np.mean(accuracy), "+-", np.std(accuracy))
print("GPR Precision with Gradient Boosting:", np.mean(precision), "+-", np.std(precision))
print("GPR Recall with Gradient Boosting:", np.mean(recall), "+-", np.std(recall))
print("GPR F1 score with Gradient Boosting:", np.mean(f1), "+-", np.std(f1))
print("GPR Overall Confusion Matrix with Gradient Boosting:")
print(confusion_matrix(all_y_test, all_y_pred))



print('\n\nXGBoost')
from xgboost import XGBClassifier
modelXGB = XGBClassifier()
all_y_test, all_y_pred = [], []
accuracy, precision, recall, f1 = [], [], [], []
for fold, (train_index, test_index) in enumerate(skf.split(X, y)):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    modelXGB.fit(X_train, y_train)
    y_pred = modelXGB.predict(X_test)
    all_y_test.extend(y_test)
    all_y_pred.extend(y_pred)
    accuracy.append(accuracy_score(y_test, y_pred))
    precision.append(precision_score(y_test, y_pred, average='macro'))
    recall.append(recall_score(y_test, y_pred, average='macro'))
    f1.append(f1_score(y_test, y_pred, average='macro'))
# Display results
print("GPR Accuracy with XGBoost:", np.mean(accuracy), "+-", np.std(accuracy))
print("GPR Precision with XGBoost:", np.mean(precision), "+-", np.std(precision))
print("GPR Recall with XGBoost:", np.mean(recall), "+-", np.std(recall))
print("GPR F1 score with XGBoost:", np.mean(f1), "+-", np.std(f1))
print("GPR Overall Confusion Matrix with XGBoost:")
print(confusion_matrix(all_y_test, all_y_pred))




import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, EarlyStopping # Added imports


X = data_set.copy()
y = labels.copy()

# Reshape X to (:, :, 1) to match the CNN input format
X = X.reshape(X.shape[0], X.shape[1], 1)

# Normalize the pixel values to be between 0 and 1 (if the data is not already scaled)
X = X.astype('float32') / 255.0

# Convert y into a one-hot encoded format
num_classes = len(set(y))  # Number of unique classes in y
y_onehot = to_categorical(y, num_classes)  # One-hot encode y

# Define the CNN model
model = models.Sequential()

# Add the first convolutional layer with 1D convolution
model.add(layers.Conv1D(32, 3, activation='relu', input_shape=(X.shape[1], 1)))
model.add(layers.MaxPooling1D(pool_size=2))

# Add another convolutional layer
model.add(layers.Conv1D(64, 3, activation='relu'))
model.add(layers.MaxPooling1D(pool_size=2))

# Add a third convolutional layer
model.add(layers.Conv1D(128, 3, activation='relu'))
model.add(layers.MaxPooling1D(pool_size=2))

# Flatten the output from the convolutional layers to pass it to the dense layers
model.add(layers.Flatten())

# Fully connected layer with 128 neurons
model.add(layers.Dense(128, activation='relu'))

# Add a dropout layer to prevent overfitting
model.add(layers.Dropout(0.5))

# Output layer (softmax for multi-class classification)
model.add(layers.Dense(num_classes, activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy', tf.keras.metrics.F1Score(average='macro')])

# Print the model summary
model.summary()

all_y_test, all_y_pred = [], []
accuracy, precision, recall, f1 = [], [], [], []
fold_count = 0 # Renamed to avoid conflict with loop variable 'fold'
for train_index, test_index in skf.split(X, y):
    fold_count+=1
    print("Fold",fold_count)
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y_onehot[train_index], y_onehot[test_index]

    # Define callbacks for the current fold
    checkpoint = ModelCheckpoint(filepath=f'best_model_fold_{fold_count}.keras', monitor='val_accuracy', save_best_only=True, mode='max', verbose=0)
    early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=0)

    model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test), callbacks=[checkpoint, early_stop]) # Modified callbacks

    # Load the best model saved by the ModelCheckpoint for the current fold
    best_model = tf.keras.models.load_model(f'best_model_fold_{fold_count}.keras') # Added this line

    # Evaluate the model
    print("\nEvaluate the model:")
    scores = best_model.evaluate(X_test, y_test) # Changed model to best_model
    y_pred = best_model.predict(X_test) # Changed model to best_model
    y_pred = np.argmax(y_pred, axis = 1)
    y_test_labels = np.argmax(y_test, axis=1) # Renamed y_val to y_test_labels for clarity
    all_y_test.extend(y_test_labels)
    all_y_pred.extend(y_pred)
    accuracy.append(accuracy_score(y_test_labels, y_pred))
    precision.append(precision_score(y_test_labels, y_pred, average='macro',zero_division=0))
    recall.append(recall_score(y_test_labels, y_pred, average='macro'))
    f1.append(f1_score(y_test_labels, y_pred, average='macro'))
# Display results
print("GPR Accuracy with CNN:", np.mean(accuracy), "+-", np.std(accuracy))
print("GPR Precision with CNN:", np.mean(precision), "+-", np.std(precision))
print("GPR Recall with CNN:", np.mean(recall), "+-", np.std(recall))
print("GPR F1 score with CNN:", np.mean(f1), "+-", np.std(f1))
print("GPR Overall Confusion Matrix with CNN:")
print(confusion_matrix(all_y_test, all_y_pred))

(11, 7, 512, 108)
(11, 7, 512, 108)
(11, 7, 512, 48)
Shape of the loaded data before rotation is (11, 7, 512, 264)
Shape of the loaded data after rotation is (264, 11, 7, 512)
Shape of the data after flipping augmentation is (1056, 11, 7, 512)
Shape of the features data is (1056, 632)
X Shape: (1056, 632)
y Shape: (1056,)


Logistic Regression


KeyboardInterrupt: 